# baza500 — FAQ Scraper: 5 polskich sklepów meblowych

**Cel:** Zebranie 500 unikalnych par pytanie/odpowiedź z kart FAQ 5 (docelowo 20) sklepów meblowych w Polsce.

## Zbadane sklepy i struktury HTML

| Sklep | URL FAQ | Pytanie (selektor) | Odpowiedź (selektor) |
|---|---|---|---|
| **Mebligo** | `/content/9-najczesciej-zadawane-pytania-faq` | `div[id^="ac-"] button[id^="ac-trigger-"]` | następne rodzeństwo `div` w bloku akordeonu |
| **Stolar Meble** | `/faq/` | `button` w `div#content` | następny element po `button` |
| **MeblujemyDOM** | `/pl/help/faq-najczesciej-zadawane-pytania-2` | elementy z `?` w `div#content` (nie-linki) | kolejne bloki tekstowe do następnego `?` |
| **SalonMeblowy.net** | `/faq.ehtml` | `h2`, `h3` w `div#afaq` | kolejne `p`/`div` do następnego nagłówka |
| **MebleM4** | `/faq-najczesciej-zadawane-pytania-w-meblem4-pl,p39.html` | elementy pasujące do `^\d+\.\s+` | kolejne bloki tekstowe do następnego numeru |


## Instalacja zależności

In [ ]:
pip install requests beautifulsoup4 pandas rapidfuzz

## Importy i konfiguracja

In [3]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def get_soup(url: str) -> BeautifulSoup:
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

## 1. Mebligo.pl

**Struktura:** akordeon `div[id^="ac-N"]` → `button[id^="ac-trigger-N"]` (pytanie) + ukryty `div` z odpowiedzią (obecny w HTML mimo CSS display:none).

In [13]:
!pip install requests_html beautifulsoup4


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\USER\anaconda3\python.exe -m pip install --upgrade pip


In [16]:
import sys
print(sys.executable)
print(sys.path)

c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\python.exe
['c:\\Users\\USER\\AppData\\Local\\Python\\pythoncore-3.14-64\\python314.zip', 'c:\\Users\\USER\\AppData\\Local\\Python\\pythoncore-3.14-64\\DLLs', 'c:\\Users\\USER\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib', 'c:\\Users\\USER\\AppData\\Local\\Python\\pythoncore-3.14-64', '', 'C:\\Users\\USER\\AppData\\Roaming\\Python\\Python314\\site-packages', 'c:\\Users\\USER\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages']


In [18]:
%pip install requests beautifulsoup4


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import requests
from bs4 import BeautifulSoup
import re


def parse_mebligo() -> list:
    url = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"
    
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    qas = []

    for block in soup.find_all("div", id=re.compile(r"^ac-\d+$")):
        btn = block.find("button", id=re.compile(r"^ac-trigger-"))
        if not btn:
            continue
        question = btn.get_text(" ", strip=True).rstrip(" +").strip()

        answer_parts = []
        for child in block.children:
            if hasattr(child, "name") and child.name and child != btn.parent:
                text = child.get_text(" ", strip=True)
                if text:
                    answer_parts.append(text)
        answer = " ".join(answer_parts).strip()

        if question and answer:
            qas.append({"shop": "Mebligo", "source_url": url,
                        "question": question, "answer": answer})
    return qas


r1 = parse_mebito()
print(f"Mebligo: {len(r1)} Q&A")
r1[:2]

Mebligo: 0 Q&A


[]

In [25]:
import requests
from bs4 import BeautifulSoup
from collections import Counter


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup(url: str) -> BeautifulSoup:
    resp = requests.get(url)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def find_candidate_blocks(soup: BeautifulSoup, min_occurrences: int = 3):
    """
    Heurystyka: znajdź <div> z klasami powtarzającymi się >= min_occurrences.
    To typowo odpowiada sekcjom typu 'faq-item', 'accordion-item' etc.
    """
    class_lists = []
    for div in soup.find_all("div"):
        classes = tuple(sorted(div.get("class", [])))
        if classes:
            class_lists.append(classes)

    counts = Counter(class_lists)
    frequent_classes = {
        cls for cls, cnt in counts.items() if cnt >= min_occurrences
    }

    candidates = []
    for div in soup.find_all("div"):
        classes = tuple(sorted(div.get("class", [])))
        if classes in frequent_classes:
            candidates.append(div)

    return candidates, frequent_classes


def debug_candidates():
    soup = get_soup(URL)
    candidates, frequent_classes = find_candidate_blocks(soup)

    print("Najczęściej powtarzające się kombinacje klas DIV:")
    for cls in frequent_classes:
        print("  ", cls)

    print("\nPrzykładowe bloki (przycięty tekst):\n")
    for i, div in enumerate(candidates[:10], start=1):
        text = div.get_text(" ", strip=True)
        print(f"--- BLOCK {i} ---")
        print(text[:500])  # pierwsze 500 znaków
        print()


if __name__ == "__main__":
    debug_candidates()

Najczęściej powtarzające się kombinacje klas DIV:
   ('ac',)
   ('ets_mm_block_content',)
   ('clearfix', 'hidden-md-up', 'title')
   ('row',)
   ('ets_mm_block', 'mm_block_type_image')
   ('ets_mm_block', 'mm_block_type_category')
   ('ac-panel',)
   ('container',)
   ('col-md-4', 'wrapper')
   ('clearfix',)

Przykładowe bloki (przycięty tekst):

--- BLOCK 1 ---
Łatwe zwroty do 30 dni od zakupu Polski producent i polski dostawca Dostawa z wniesieniem tel:+48 455 455 044

--- BLOCK 2 ---
Łatwe zwroty do 30 dni od zakupu Polski producent i polski dostawca Dostawa z wniesieniem tel:+48 455 455 044

--- BLOCK 3 ---
search clear  Zaloguj się shopping_cart Koszyk 0 

--- BLOCK 4 ---
search clear  Zaloguj się shopping_cart Koszyk 0 

--- BLOCK 5 ---


--- BLOCK 6 ---
search clear

--- BLOCK 7 ---
search clear

--- BLOCK 8 ---
Menu Menu Powrót Strona główna Salon Meble do salonu Sofy i kanapy Narożniki Komody Elementy dopełniające Fotele Poduszki Pufy Sprawdź najczęściej kupowane produkty

In [26]:
import re
import requests
from bs4 import BeautifulSoup


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup(url: str) -> BeautifulSoup:
    resp = requests.get(url)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def parse_mebligo() -> list:
    soup = get_soup(URL)
    qas = []

    # Znajdź wszystkie diva z id w formie ac-panel-<liczba>
    for panel in soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$")):
        panel_id = panel.get("id")
        m = re.match(r"ac-panel-(\d+)", panel_id)
        if not m:
            continue
        idx = m.group(1)

        # Spróbuj znaleźć odpowiadające pytanie po id ac-trigger-<liczba>
        trigger = soup.find(id=f"ac-trigger-{idx}")

        # Jeżeli trigger nie istnieje, spróbuj wziąć poprzedni nagłówek/rodzica
        question_text = None
        if trigger:
            question_text = trigger.get_text(" ", strip=True)
        else:
            # fallback: spróbuj znaleźć pytanie jako poprzedni nagłówek
            prev_heading = panel.find_previous(["h2", "h3", "button"])
            if prev_heading:
                question_text = prev_heading.get_text(" ", strip=True)

        answer_text = panel.get_text(" ", strip=True)

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": URL,
                    "question": question_text,
                    "answer": answer_text,
                }
            )

    return qas


if __name__ == "__main__":
    r1 = parse_mebligo()
    print(f"Mebligo: {len(r1)} Q&A")
    for qa in r1[:5]:
        print("Q:", qa["question"])
        print("A:", qa["answer"])
        print("-" * 40)

Mebligo: 0 Q&A


In [28]:
from requests_html import HTMLSession
from bs4 import BeautifulSoup
import re


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup(url: str) -> BeautifulSoup:
    session = HTMLSession()
    resp = session.get(url)
    html_text = resp.html.html  # pełen HTML z requests_html
    session.close()
    return BeautifulSoup(html_text, "html.parser")


def parse_mebligo() -> list:
    soup = get_soup(URL)
    qas = []

    for panel in soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$")):
        panel_id = panel.get("id")
        m = re.match(r"ac-panel-(\d+)", panel_id)
        if not m:
            continue
        idx = m.group(1)

        trigger = soup.find(id=f"ac-trigger-{idx}")

        question_text = None
        if trigger:
            question_text = trigger.get_text(" ", strip=True)
        else:
            prev_heading = panel.find_previous(["h2", "h3", "button"])
            if prev_heading:
                question_text = prev_heading.get_text(" ", strip=True)

        answer_text = panel.get_text(" ", strip=True)

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": URL,
                    "question": question_text,
                    "answer": answer_text,
                }
            )

    return qas


r1 = parse_mebligo()
print(f"Mebligo: {len(r1)} Q&A")
r1[:5]

ModuleNotFoundError: No module named 'requests_html'

Niestety dupa! zmieniamy metodologie

In [29]:
import re
import requests
from bs4 import BeautifulSoup


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup(url: str) -> BeautifulSoup:
    resp = requests.get(url)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def parse_mebligo() -> list:
    soup = get_soup(URL)
    qas = []

    # Wszystkie panele odpowiedzi: <div id="ac-panel-1">, <div id="ac-panel-2">, ...
    for panel in soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$")):
        panel_id = panel.get("id") or ""
        m = re.match(r"ac-panel-(\d+)", panel_id)
        if not m:
            continue
        idx = m.group(1)

        # Spróbuj znaleźć pytanie po id="ac-trigger-N"
        trigger = soup.find(id=f"ac-trigger-{idx}")

        question_text = None
        if trigger:
            question_text = trigger.get_text(" ", strip=True)
        else:
            # fallback: poprzedni nagłówek (np. h2/h3/button) przed panelem
            prev_heading = panel.find_previous(["h2", "h3", "button"])
            if prev_heading:
                question_text = prev_heading.get_text(" ", strip=True)

        answer_text = panel.get_text(" ", strip=True)

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": URL,
                    "question": question_text,
                    "answer": answer_text,
                }
            )

    return qas


def debug_panels(limit: int = 3):
    soup = get_soup(URL)
    panels = soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$"))
    print(f"Znaleziono {len(panels)} paneli ac-panel-*")
    for i, panel in enumerate(panels[:limit], start=1):
        print(f"--- PANEL {i} ({panel.get('id')}) ---")
        print(panel.prettify()[:1000])
        print()


# Najpierw podejrzyj, czy w ogóle widzimy ac-panel-*
debug_panels()

# Potem uruchom właściwy parser
r1 = parse_mebligo()
print(f"Mebligo: {len(r1)} Q&A")
r1[:5]

Znaleziono 0 paneli ac-panel-*
Mebligo: 0 Q&A


[]

In [30]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import re

URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"

def get_soup_with_selenium() -> BeautifulSoup:
    options = Options()
    options.add_argument("--headless=new")
    driver = webdriver.Chrome(options=options)

    try:
        driver.get(URL)
        # tu ewentualnie: poczekaj na elementy FAQ, np. WebDriverWait(...)
        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def parse_mebligo() -> list:
    soup = get_soup_with_selenium()
    qas = []

    # teraz SELENIUM widzi to samo co DevTools, więc istnieją ac-panel-*
    for panel in soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$")):
        panel_id = panel.get("id") or ""
        m = re.match(r"ac-panel-(\d+)", panel_id)
        if not m:
            continue
        idx = m.group(1)

        trigger = soup.find(id=f"ac-trigger-{idx}")

        question_text = None
        if trigger:
            question_text = trigger.get_text(" ", strip=True)
        else:
            prev_heading = panel.find_previous(["h2", "h3", "button"])
            if prev_heading:
                question_text = prev_heading.get_text(" ", strip=True)

        answer_text = panel.get_text(" ", strip=True)

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": URL,
                    "question": question_text,
                    "answer": answer_text,
                }
            )

    return qas

ModuleNotFoundError: No module named 'selenium'

In [31]:
%pip install selenium

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 3.4 MB/s eta 0:00:03
   ------ --------------------------------- 1.6/9.7 MB 4.2 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.7 MB 4.5 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.7 MB 4.6 MB/s eta 0:00:02
   ------------------- -------------------- 4.7/9.7 MB 4.6 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.7 MB 4.7 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.7 MB 4.7 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.7 MB 4.7 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.7 MB 4.7 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 4.6 MB/s  0:00:02

   ----------------------------------------  0/12 [sortedcontainers]
   --- ------------------------------------  1/12 [websocket-client]
   --- ----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
%pip install chromedriver-autoinstaller

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
%pip install chromedriver-autoinstaller

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import re
import time

import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import requests  # może się przydać gdzie indziej


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup_with_selenium() -> BeautifulSoup:
    # Upewnij się, że chromedriver jest zainstalowany
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=options)

    try:
        driver.get(URL)

        # Prosty wait – możesz później zastąpić WebDriverWait
        time.sleep(3)

        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def parse_mebligo() -> list:
    soup = get_soup_with_selenium()
    qas = []

    # Wszystkie panele odpowiedzi: <div id="ac-panel-1">, <div id="ac-panel-2">, ...
    panels = soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$"))
    print(f"Znaleziono {len(panels)} paneli ac-panel-*")

    for panel in panels:
        panel_id = panel.get("id") or ""
        m = re.match(r"ac-panel-(\d+)", panel_id)
        if not m:
            continue
        idx = m.group(1)

        # Pytanie – spróbujmy id="ac-trigger-<n>"
        trigger = soup.find(id=f"ac-trigger-{idx}")

        question_text = None
        if trigger:
            question_text = trigger.get_text(" ", strip=True)
        else:
            # fallback: poprzedni nagłówek lub button
            prev_heading = panel.find_previous(["h2", "h3", "button"])
            if prev_heading:
                question_text = prev_heading.get_text(" ", strip=True)

        answer_text = panel.get_text(" ", strip=True)

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": URL,
                    "question": question_text,
                    "answer": answer_text,
                }
            )

    return qas


r1 = parse_mebligo()
print(f"Mebligo: {len(r1)} Q&A")
r1[:5]

Znaleziono 45 paneli ac-panel-*
Mebligo: 45 Q&A


[{'shop': 'Mebligo',
  'source_url': 'https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq',
  'question': 'Czy można gdzieś zobaczyć meble przed dostawą/usiąść na nich?',
  'answer': 'Niestety, nie posiadamy stacjonarnych salonów sprzedaży.'},
 {'shop': 'Mebligo',
  'source_url': 'https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq',
  'question': 'Czy jest możliwość modyfikacji mebla (zmiana wymiarów, koloru)?',
  'answer': 'Niestety, nie produkujemy mebli na wymiar. W wybranych modelach mebli można konfigurować dostępne parametry.'},
 {'shop': 'Mebligo',
  'source_url': 'https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq',
  'question': 'Gdzie znajdę instrukcję montażu?',
  'answer': 'Instrukcje montażu naszych mebli otrzymasz w smsie informującym o etapie realizacji Twojego zamówienia. Znajdziesz je też na stronie naszego sklepu, w produktach w zakładce Instrukcje.'},
 {'shop': 'Mebligo',
  'source_url': 'https://mebligo.pl/content/9-najczesciej-zadawa

## Panel djagnostyczny

In [11]:
import re
import time
import random
from typing import List, Dict

import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup_with_selenium(url: str) -> BeautifulSoup:
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, timeout=15)

    try:
        driver.get(url)
        time.sleep(2)

        try:
            accordion_panels = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[id^='ac-']")
                )
            )
            print(f"Znaleziono {len(accordion_panels)} paneli akordeonu (Selenium)")
        except TimeoutException:
            print("Akordeon nie został znaleziony, kontynuuję bez niego")

        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def debug_panel_full(panel, idx: int, final_answer: str, final_question: str):
    panel_id = panel.get("id", "")
    print("\n" + "=" * 80)
    print(f"[DEBUG FULL] PANEL {idx}  |  id={panel_id}")
    print("=" * 80)

    raw_html = str(panel)
    print("\n[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):")
    max_len = 2000
    print(raw_html[:max_len] + ("..." if len(raw_html) > max_len else ""))

    full_text = clean_text(panel.get_text(" ", strip=True))
    print("\n[FULL TEXT Z PANELU]:")
    print(full_text)

    div_p_texts: List[str] = []
    for node in panel.find_all(["div", "p"]):
        t = clean_text(node.get_text(" ", strip=True))
        if t:
            div_p_texts.append(t)
    print("\n[SUMA TEKSTÓW z <div>/<p>]:")
    print(" | ".join(div_p_texts) if div_p_texts else "(brak)")

    panel_copy = BeautifulSoup(str(panel), "html.parser")
    for btn in panel_copy.find_all("button"):
        btn.decompose()
    no_button_text = clean_text(panel_copy.get_text(" ", strip=True))
    print("\n[TEKST PANELU bez <button>]:")
    print(no_button_text if no_button_text else "(brak)")

    print("\n[AKTUALNA INTERPRETACJA]:")
    print(f"Q: {final_question}")
    print(f"A: {final_answer}")

    print("\n" + "-" * 80)
    print("[KONIEC PEŁNEGO DEBUGU PANELU]")
    print("-" * 80 + "\n")


def extract_answer_from_block(block) -> str:
    """
    Docelowy ekstraktor:
    - w bloku ac-N szukamy div.ac-panel-N
    - bierzemy tekst ze wszystkich <p> w środku (deduplikacja)
    - fallback: cały panel bez <button>, gdyby struktura się zmieniła
    """
    panel = block.find("div", id=re.compile(r"^ac-panel-\d+$"))
    if panel:
        parts: List[str] = []
        for p in panel.find_all("p"):
            t = clean_text(p.get_text(" ", strip=True))
            if t:
                parts.append(t)
        if parts:
            # deduplikacja
            seen = set()
            uniq = []
            for t in parts:
                if t not in seen:
                    seen.add(t)
                    uniq.append(t)
            return " ".join(uniq)

    # fallback: cały block bez buttonów
    block_copy = BeautifulSoup(str(block), "html.parser")
    for btn in block_copy.find_all("button"):
        btn.decompose()
    return clean_text(block_copy.get_text(" ", strip=True))


def parse_mebligo_with_diagnostics(url: str = URL, debug_count: int = 5) -> List[Dict]:
    soup = get_soup_with_selenium(url)
    qas: List[Dict] = []

    blocks = soup.find_all("div", id=re.compile(r"^ac-\d+$"))
    total = len(blocks)
    print(f"\nZnaleziono {total} bloków akordeonu (ac-0, ac-1, ...)")

    if total == 0:
        return qas

    debug_indices = sorted(
        random.sample(range(1, total + 1), k=min(debug_count, total))
    )
    print(f"\nPanele wybrane do PEŁNEJ DIAGNOSTYKI: {debug_indices}")

    for idx, block in enumerate(blocks, start=1):
        block_id = block.get("id", "")

        trigger = block.find("button", id=re.compile(r"^ac-trigger-\d+$"))
        if trigger:
            question_text = clean_text(trigger.get_text(" ", strip=True)).rstrip(" +").strip()
        else:
            header = block.find("h2", class_="ac-header") or block.find("h3")
            question_text = clean_text(header.get_text(" ", strip=True)) if header else ""

        answer_text = extract_answer_from_block(block)

        if idx in debug_indices:
            debug_panel_full(block, idx, answer_text, question_text)

        print(f"Panel {idx}: id={block_id}")
        print(f"  Q: {question_text[:70] + '...' if question_text else 'BRAK'}")
        print(f"  A_len: {len(answer_text)}")

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": url,
                    "question": question_text,
                    "answer": answer_text,
                    "question_index": idx,
                    "panel_id": block_id,
                    "position": f"accordion[{idx}]",
                }
            )

    print(f"\nMebligo: {len(qas)} Q&A pobrane (diagnostics + final)")
    return qas


if __name__ == "__main__":
    r1 = parse_mebligo_with_diagnostics(debug_count=5)
    print(f"\nŁącznie Q&A: {len(r1)}")
    if r1:
        print("\nPierwsze 3 pytania:")
        for i, qa in enumerate(r1[:3], start=1):
            print(f"\n{i}. {qa['question']}")
            print(f"   Odpowiedź: {qa['answer'][:200]}...")

Znaleziono 90 paneli akordeonu (Selenium)

Znaleziono 45 bloków akordeonu (ac-0, ac-1, ...)

Panele wybrane do PEŁNEJ DIAGNOSTYKI: [1, 20, 32, 42, 45]

[DEBUG FULL] PANEL 1  |  id=ac-0

[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):
<div class="ac js-enabled" id="ac-0">
<h2 class="ac-header"><button aria-controls="ac-panel-0" aria-disabled="false" aria-expanded="false" class="ac-trigger" id="ac-trigger-0" role="button" type="button">Czy można gdzieś zobaczyć meble przed dostawą/usiąść na nich?</button></h2>
<div aria-labelledby="ac-trigger-0" class="ac-panel" id="ac-panel-0" role="region" style="transition-duration: 600ms; height: 0px;">
<p>Niestety, nie posiadamy stacjonarnych salonów sprzedaży.</p>
</div>
</div>

[FULL TEXT Z PANELU]:
Czy można gdzieś zobaczyć meble przed dostawą/usiąść na nich? Niestety, nie posiadamy stacjonarnych salonów sprzedaży.

[SUMA TEKSTÓW z <div>/<p>]:
Niestety, nie posiadamy stacjonarnych salonów sprzedaży. | Niestety, nie posiadamy stacjonarnych salo

Wersja poprawiona ale nie widzę tego.

In [12]:
import re
import time
import random
from typing import List, Dict

import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup_with_selenium(url: str) -> BeautifulSoup:
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, timeout=15)

    try:
        driver.get(url)
        time.sleep(2)

        try:
            accordion_panels = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[id^='ac-']")
                )
            )
            print(f"Znaleziono {len(accordion_panels)} paneli akordeonu (Selenium)")
        except TimeoutException:
            print("Akordeon nie został znaleziony, kontynuuję bez niego")

        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def debug_panel_full(panel, idx: int, final_answer: str, final_question: str):
    panel_id = panel.get("id", "")
    print("\n" + "=" * 80)
    print(f"[DEBUG FULL] PANEL {idx}  |  id={panel_id}")
    print("=" * 80)

    raw_html = str(panel)
    print("\n[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):")
    max_len = 2000
    print(raw_html[:max_len] + ("..." if len(raw_html) > max_len else ""))

    full_text = clean_text(panel.get_text(" ", strip=True))
    print("\n[FULL TEXT Z PANELU]:")
    print(full_text)

    div_p_texts: List[str] = []
    for node in panel.find_all(["div", "p"]):
        t = clean_text(node.get_text(" ", strip=True))
        if t:
            div_p_texts.append(t)
    print("\n[SUMA TEKSTÓW z <div>/<p>]:")
    print(" | ".join(div_p_texts) if div_p_texts else "(brak)")

    panel_copy = BeautifulSoup(str(panel), "html.parser")
    for btn in panel_copy.find_all("button"):
        btn.decompose()
    no_button_text = clean_text(panel_copy.get_text(" ", strip=True))
    print("\n[TEKST PANELU bez <button>]:")
    print(no_button_text if no_button_text else "(brak)")

    print("\n[AKTUALNA INTERPRETACJA]:")
    print(f"Q: {final_question}")
    print(f"A: {final_answer}")

    print("\n" + "-" * 80)
    print("[KONIEC PEŁNEGO DEBUGU PANELU]")
    print("-" * 80 + "\n")


def extract_answer_from_block(block) -> str:
    """
    Docelowy ekstraktor:
    - w bloku ac-N szukamy div.ac-panel-N
    - bierzemy tekst ze wszystkich <p> w środku (deduplikacja)
    - fallback: cały panel bez <button>, gdyby struktura się zmieniła
    """
    panel = block.find("div", id=re.compile(r"^ac-panel-\d+$"))
    if panel:
        parts: List[str] = []
        for p in panel.find_all("p"):
            t = clean_text(p.get_text(" ", strip=True))
            if t:
                parts.append(t)
        if parts:
            # deduplikacja
            seen = set()
            uniq = []
            for t in parts:
                if t not in seen:
                    seen.add(t)
                    uniq.append(t)
            return " ".join(uniq)

    # fallback: cały block bez buttonów
    block_copy = BeautifulSoup(str(block), "html.parser")
    for btn in block_copy.find_all("button"):
        btn.decompose()
    return clean_text(block_copy.get_text(" ", strip=True))


def parse_mebligo_with_diagnostics(url: str = URL, debug_count: int = 5) -> List[Dict]:
    soup = get_soup_with_selenium(url)
    qas: List[Dict] = []

    blocks = soup.find_all("div", id=re.compile(r"^ac-\d+$"))
    total = len(blocks)
    print(f"\nZnaleziono {total} bloków akordeonu (ac-0, ac-1, ...)")

    if total == 0:
        return qas

    debug_indices = sorted(
        random.sample(range(1, total + 1), k=min(debug_count, total))
    )
    print(f"\nPanele wybrane do PEŁNEJ DIAGNOSTYKI: {debug_indices}")

    for idx, block in enumerate(blocks, start=1):
        block_id = block.get("id", "")

        trigger = block.find("button", id=re.compile(r"^ac-trigger-\d+$"))
        if trigger:
            question_text = clean_text(trigger.get_text(" ", strip=True)).rstrip(" +").strip()
        else:
            header = block.find("h2", class_="ac-header") or block.find("h3")
            question_text = clean_text(header.get_text(" ", strip=True)) if header else ""

        answer_text = extract_answer_from_block(block)

        if idx in debug_indices:
            debug_panel_full(block, idx, answer_text, question_text)

        print(f"Panel {idx}: id={block_id}")
        print(f"  Q: {question_text[:70] + '...' if question_text else 'BRAK'}")
        print(f"  A_len: {len(answer_text)}")

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": url,
                    "question": question_text,
                    "answer": answer_text,
                    "question_index": idx,
                    "panel_id": block_id,
                    "position": f"accordion[{idx}]",
                }
            )

    print(f"\nMebligo: {len(qas)} Q&A pobrane (diagnostics + final)")
    return qas


if __name__ == "__main__":
    r1 = parse_mebligo_with_diagnostics(debug_count=5)
    print(f"\nŁącznie Q&A: {len(r1)}")
    if r1:
        print("\nPierwsze 3 pytania:")
        for i, qa in enumerate(r1[:3], start=1):
            print(f"\n{i}. {qa['question']}")
            print(f"   Odpowiedź: {qa['answer'][:200]}...")

Znaleziono 90 paneli akordeonu (Selenium)

Znaleziono 45 bloków akordeonu (ac-0, ac-1, ...)

Panele wybrane do PEŁNEJ DIAGNOSTYKI: [9, 33, 34, 36, 45]
Panel 1: id=ac-0
  Q: Czy można gdzieś zobaczyć meble przed dostawą/usiąść na nich?...
  A_len: 56
Panel 2: id=ac-1
  Q: Czy jest możliwość modyfikacji mebla (zmiana wymiarów, koloru)?...
  A_len: 108
Panel 3: id=ac-2
  Q: Gdzie znajdę instrukcję montażu?...
  A_len: 184
Panel 4: id=ac-3
  Q: Jakie są wymiary paczek?...
  A_len: 161
Panel 5: id=ac-4
  Q: Mebel będzie dostarczany w całości czy paczkach do samodzielnego monta...
  A_len: 259
Panel 6: id=ac-5
  Q: Jak mogę określić stronę mebla?...
  A_len: 196
Panel 7: id=ac-6
  Q: Jakie wypełnienie jest bardziej komfortowe?...
  A_len: 891
Panel 8: id=ac-7
  Q: Co to znaczy, że narożnik jest uniwersalny?...
  A_len: 253

[DEBUG FULL] PANEL 9  |  id=ac-8

[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):
<div class="ac js-enabled" id="ac-8">
<h2 class="ac-header"><button aria-controls="ac

# Dobry kod tworzy bazę

In [13]:
import re
import time
import random
from typing import List, Dict

import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup_with_selenium(url: str) -> BeautifulSoup:
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, timeout=15)

    try:
        driver.get(url)
        time.sleep(2)

        try:
            accordion_panels = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[id^='ac-']")
                )
            )
            print(f"Znaleziono {len(accordion_panels)} paneli akordeonu (Selenium)")
        except TimeoutException:
            print("Akordeon nie został znaleziony, kontynuuję bez niego")

        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def extract_question(block) -> str:
    trigger = block.find("button", id=re.compile(r"^ac-trigger-\d+$"))
    if trigger:
        return clean_text(trigger.get_text(" ", strip=True)).rstrip(" +").strip()
    heading = block.find("h2") or block.find("h3")
    if heading:
        return clean_text(heading.get_text(" ", strip=True))
    return ""


def extract_answer(block) -> str:
    """
    Docelowa logika pod aktualną strukturę:
    - najpierw szukamy div#ac-panel-N
    - z niego bierzemy wszystkie <p> (deduplikacja)
    - jeśli brak <p>, bierzemy cały panel
    - jeśli nie ma panelu, fallback: cały blok bez <button>
    """
    panel = block.find("div", id=re.compile(r"^ac-panel-\d+$"))
    if panel:
        parts: List[str] = []
        for p in panel.find_all("p"):
            t = clean_text(p.get_text(" ", strip=True))
            if t:
                parts.append(t)
        if parts:
            seen = set()
            uniq = []
            for t in parts:
                if t not in seen:
                    seen.add(t)
                    uniq.append(t)
            return " ".join(uniq)
        # fallback: cały panel
        return clean_text(panel.get_text(" ", strip=True))

    # fallback: cały block bez buttonów
    block_copy = BeautifulSoup(str(block), "html.parser")
    for btn in block_copy.find_all("button"):
        btn.decompose()
    return clean_text(block_copy.get_text(" ", strip=True))


def debug_panel_full(panel, idx: int):
    """
    Pełna diagnostyka JEDNEGO panelu:
    - RAW HTML
    - FULL TEXT
    - TEXT z <div>/<p>
    - TEXT bez <button>
    - aktualne pytanie i odpowiedź wg naszej logiki
    """
    panel_id = panel.get("id", "")
    print("\n" + "=" * 80)
    print(f"[DEBUG FULL] PANEL {idx}  |  id={panel_id}")
    print("=" * 80)

    # RAW HTML
    raw_html = str(panel)
    print("\n[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):")
    max_len = 2000
    print(raw_html[:max_len] + ("..." if len(raw_html) > max_len else ""))

    # FULL TEXT
    full_text = clean_text(panel.get_text(" ", strip=True))
    print("\n[FULL TEXT Z PANELU]:")
    print(full_text)

    # TEKSTY z <div>/<p>
    div_p_texts: List[str] = []
    for node in panel.find_all(["div", "p"]):
        t = clean_text(node.get_text(" ", strip=True))
        if t:
            div_p_texts.append(t)
    print("\n[SUMA TEKSTÓW z <div>/<p>]:")
    print(" | ".join(div_p_texts) if div_p_texts else "(brak)")

    # TEKST PANELU bez <button>
    panel_copy = BeautifulSoup(str(panel), "html.parser")
    for btn in panel_copy.find_all("button"):
        btn.decompose()
    no_button_text = clean_text(panel_copy.get_text(" ", strip=True))
    print("\n[TEKST PANELU bez <button>]:")
    print(no_button_text if no_button_text else "(brak)")

    # Aktualna logika pytanie/odpowiedź (ta sama co do CSV)
    final_question = extract_question(panel)
    final_answer = extract_answer(panel)

    print("\n[AKTUALNA INTERPRETACJA]:")
    print(f"Q: {final_question}")
    print(f"A: {final_answer}")

    print("\n" + "-" * 80)
    print("[KONIEC PEŁNEGO DEBUGU PANELU]")
    print("-" * 80 + "\n")


def run_full_diagnostics_and_build_csv(
    url: str = URL, sample_size: int = 5, csv_path: str = "mebligo_faq.csv"
) -> List[Dict]:
    """
    1) Diagnostyka: 5 losowych paneli (pełny debug).
    2) Budowa bazy Q&A z całej strony.
    3) Zapis do CSV.
    """
    soup = get_soup_with_selenium(url)

    blocks = soup.find_all("div", id=re.compile(r"^ac-\d+$"))
    total = len(blocks)
    print(f"\nZnaleziono {total} bloków akordeonu (ac-0, ac-1, ...)")

    if total == 0:
        print("Brak bloków do przetworzenia.")
        return []

    # LOSOWE PANELE DO DIAGNOZY
    sample_size = min(sample_size, total)
    indices = sorted(random.sample(range(1, total + 1), k=sample_size))
    print(f"\nPanele wybrane do PEŁNEJ DIAGNOSTYKI: {indices}")

    # DIAGNOSTYKA
    for idx in indices:
        panel = blocks[idx - 1]
        debug_panel_full(panel, idx)

    # BUDOWA BAZY Q&A
    qas: List[Dict] = []
    for idx, block in enumerate(blocks, start=1):
        block_id = block.get("id", "")

        question = extract_question(block)
        answer = extract_answer(block)

        print(f"Panel {idx}: id={block_id}")
        print(f"  Q: {question[:70] + '...' if question else 'BRAK'}")
        print(f"  A_len: {len(answer)}")

        if question and answer:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": url,
                    "question": question,
                    "answer": answer,
                    "question_index": idx,
                    "panel_id": block_id,
                    "position": f"accordion[{idx}]",
                }
            )

    print(f"\nMebligo: {len(qas)} Q&A zebrane")

    # ZAPIS DO CSV
    if qas:
        df = pd.DataFrame(qas)
        df.to_csv(csv_path, index=False, encoding="utf-8")
        print(f"Zapisano do pliku CSV: {csv_path}")

    return qas


if __name__ == "__main__":
    qas = run_full_diagnostics_and_build_csv(sample_size=5, csv_path="mebligo_faq.csv")
    print(f"\nŁącznie Q&A w bazie: {len(qas)}")

Znaleziono 90 paneli akordeonu (Selenium)

Znaleziono 45 bloków akordeonu (ac-0, ac-1, ...)

Panele wybrane do PEŁNEJ DIAGNOSTYKI: [8, 15, 33, 36, 40]

[DEBUG FULL] PANEL 8  |  id=ac-7

[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):
<div class="ac js-enabled" id="ac-7">
<h2 class="ac-header"><button aria-controls="ac-panel-7" aria-disabled="false" aria-expanded="false" class="ac-trigger" id="ac-trigger-7" role="button" type="button">Co to znaczy, że narożnik jest uniwersalny?</button></h2>
<div aria-labelledby="ac-trigger-7" class="ac-panel" id="ac-panel-7" role="region" style="transition-duration: 600ms; height: 0px;">
<p>Narożnik uniwersalny to produkt, którego stronę określasz podczas pierwszego montażu. To Ty decydujesz, która strona mebla wpasuje się w Twoją przestrzeń. Jednak po dokonaniu pierwszego montażu mebel będzie już posiadał otwory, których nie można usunąć.</p>
</div>
</div>

[FULL TEXT Z PANELU]:
Co to znaczy, że narożnik jest uniwersalny? Narożnik uniwersalny to pr

## Wersja z dopisywaniem do bazy i sprawdzeniem duplikatów przed dopisaniem

In [14]:
import re
import time
import random
from typing import List, Dict

import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd
import os


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"
SHOP_NAME = "Mebligo"
CSV_PATH = "faq_dataset.csv"   # wspólna baza dla wielu sklepów/URL-i


def get_soup_with_selenium(url: str) -> BeautifulSoup:
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, timeout=15)

    try:
        driver.get(url)
        time.sleep(2)

        try:
            accordion_panels = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[id^='ac-']")
                )
            )
            print(f"Znaleziono {len(accordion_panels)} paneli akordeonu (Selenium)")
        except TimeoutException:
            print("Akordeon nie został znaleziony, kontynuuję bez niego")

        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def extract_question(block) -> str:
    trigger = block.find("button", id=re.compile(r"^ac-trigger-\d+$"))
    if trigger:
        return clean_text(trigger.get_text(" ", strip=True)).rstrip(" +").strip()
    heading = block.find("h2") or block.find("h3")
    if heading:
        return clean_text(heading.get_text(" ", strip=True))
    return ""


def extract_answer(block) -> str:
    """
    Logika:
    - div#ac-panel-N -> tekst z <p> (deduplikacja), fallback: cały panel
    - jeśli brak panelu -> cały block bez <button>
    """
    panel = block.find("div", id=re.compile(r"^ac-panel-\d+$"))
    if panel:
        parts: List[str] = []
        for p in panel.find_all("p"):
            t = clean_text(p.get_text(" ", strip=True))
            if t:
                parts.append(t)
        if parts:
            seen = set()
            uniq = []
            for t in parts:
                if t not in seen:
                    seen.add(t)
                    uniq.append(t)
            return " ".join(uniq)
        return clean_text(panel.get_text(" ", strip=True))

    block_copy = BeautifulSoup(str(block), "html.parser")
    for btn in block_copy.find_all("button"):
        btn.decompose()
    return clean_text(block_copy.get_text(" ", strip=True))


def debug_panel_full(panel, idx: int):
    """
    Pełna diagnostyka JEDNEGO panelu.
    """
    panel_id = panel.get("id", "")
    print("\n" + "=" * 80)
    print(f"[DEBUG FULL] PANEL {idx}  |  id={panel_id}")
    print("=" * 80)

    raw_html = str(panel)
    print("\n[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):")
    max_len = 2000
    print(raw_html[:max_len] + ("..." if len(raw_html) > max_len else ""))

    full_text = clean_text(panel.get_text(" ", strip=True))
    print("\n[FULL TEXT Z PANELU]:")
    print(full_text)

    div_p_texts: List[str] = []
    for node in panel.find_all(["div", "p"]):
        t = clean_text(node.get_text(" ", strip=True))
        if t:
            div_p_texts.append(t)
    print("\n[SUMA TEKSTÓW z <div>/<p>]:")
    print(" | ".join(div_p_texts) if div_p_texts else "(brak)")

    panel_copy = BeautifulSoup(str(panel), "html.parser")
    for btn in panel_copy.find_all("button"):
        btn.decompose()
    no_button_text = clean_text(panel_copy.get_text(" ", strip=True))
    print("\n[TEKST PANELU bez <button>]:")
    print(no_button_text if no_button_text else "(brak)")

    final_question = extract_question(panel)
    final_answer = extract_answer(panel)

    print("\n[AKTUALNA INTERPRETACJA]:")
    print(f"Q: {final_question}")
    print(f"A: {final_answer}")

    print("\n" + "-" * 80)
    print("[KONIEC PEŁNEGO DEBUGU PANELU]")
    print("-" * 80 + "\n")


def scrape_mebligo(url: str = URL, shop: str = SHOP_NAME,
                   debug_sample: int = 5) -> List[Dict]:
    """
    1) Diagnostyka: losowe debug_sample paneli.
    2) Zwraca listę Q&A z tej jednej strony.
    """
    soup = get_soup_with_selenium(url)

    blocks = soup.find_all("div", id=re.compile(r"^ac-\d+$"))
    total = len(blocks)
    print(f"\nZnaleziono {total} bloków akordeonu (ac-0, ac-1, ...)")

    if total == 0:
        print("Brak bloków do przetworzenia.")
        return []

    # losowe panele do debug
    sample_size = min(debug_sample, total)
    indices = sorted(random.sample(range(1, total + 1), k=sample_size))
    print(f"\nPanele wybrane do PEŁNEJ DIAGNOSTYKI: {indices}")

    for idx in indices:
        panel = blocks[idx - 1]
        debug_panel_full(panel, idx)

    # właściwy scraping
    qas: List[Dict] = []
    for idx, block in enumerate(blocks, start=1):
        block_id = block.get("id", "")

        question = extract_question(block)
        answer = extract_answer(block)

        print(f"Panel {idx}: id={block_id}")
        print(f"  Q: {question[:70] + '...' if question else 'BRAK'}")
        print(f"  A_len: {len(answer)}")

        if question and answer:
            qas.append(
                {
                    "shop": shop,
                    "source_url": url,
                    "question": question,
                    "answer": answer,
                    "question_index": idx,
                    "panel_id": block_id,
                    "position": f"accordion[{idx}]",
                }
            )

    print(f"\n{shop}: {len(qas)} Q&A zebrane z {url}")
    return qas


def append_new_unique_rows(
    new_rows: List[Dict],
    csv_path: str = CSV_PATH,
    key_cols=("shop", "source_url", "question"),
) -> None:
    """
    Zasady:
    - jeśli CSV nie istnieje -> tworzy nowy plik z new_rows
    - jeśli istnieje:
        - wczytuje istniejącą bazę,
        - WYZNACZA, które new_rows mają już odpowiednik
          o tym samym (shop, source_url, question) -> te pomija,
        - dopisuje TYLKO nowe, unikalne rekordy.
    """
    if not new_rows:
        print("Brak nowych danych do zapisania.")
        return

    new_df = pd.DataFrame(new_rows)

    if not os.path.exists(csv_path):
        new_df.to_csv(csv_path, index=False, encoding="utf-8")
        print(f"Stworzono nowy plik CSV: {csv_path}")
        print(f"Dopisano {len(new_df)} nowych rekordów.")
        return

    existing_df = pd.read_csv(csv_path)

    # klucz logiczny w istniejącej bazie
    existing_keys = set(
        tuple(row[col] for col in key_cols)
        for _, row in existing_df.iterrows()
    )

    # filtruj tylko rekordy, których klucza jeszcze nie ma
    mask_new = []
    for _, row in new_df.iterrows():
        key = tuple(row[col] for col in key_cols)
        mask_new.append(key not in existing_keys)

    new_unique_df = new_df[mask_new]

    if new_unique_df.empty:
        print("Brak naprawdę nowych rekordów – CSV pozostaje bez zmian.")
        return

    updated = pd.concat([existing_df, new_unique_df], ignore_index=True)
    updated.to_csv(csv_path, index=False, encoding="utf-8")

    print(f"Zaktualizowano plik CSV: {csv_path}")
    print(f"Dopisano {len(new_unique_df)} nowych rekordów.")
    print(f"Łącznie rekordów w CSV: {len(updated)}")


if __name__ == "__main__":
    # 1) scrap jednej strony (Mebligo)
    qas = scrape_mebligo(URL, SHOP_NAME, debug_sample=5)

    # 2) dopisz 

Znaleziono 90 paneli akordeonu (Selenium)

Znaleziono 45 bloków akordeonu (ac-0, ac-1, ...)

Panele wybrane do PEŁNEJ DIAGNOSTYKI: [4, 9, 10, 21, 22]

[DEBUG FULL] PANEL 4  |  id=ac-3

[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):
<div class="ac js-enabled" id="ac-3">
<h2 class="ac-header"><button aria-controls="ac-panel-3" aria-disabled="false" aria-expanded="false" class="ac-trigger" id="ac-trigger-3" role="button" type="button">Jakie są wymiary paczek?</button></h2>
<div aria-labelledby="ac-trigger-3" class="ac-panel" id="ac-panel-3" role="region" style="transition-duration: 600ms; height: 0px;">
<p>Wymiary paczek zależne są od wybranego mebla. Jeśli potrzebujesz takiej informacji, skontaktuj się z nami (e-mail: kontakt@mebligo.pl, telefon: +48 455 455 044).</p>
</div>
</div>

[FULL TEXT Z PANELU]:
Jakie są wymiary paczek? Wymiary paczek zależne są od wybranego mebla. Jeśli potrzebujesz takiej informacji, skontaktuj się z nami (e-mail: kontakt@mebligo.pl, telefon: +48 455 455 0

In [ ]:
https://selsey.pl/pages/pytania-i-odpowiedzi

# Nie próbowane - pomieszany kod

In [ ]:
import re
import time
import random
from typing import List, Dict

import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd
import os


URL = "https://selsey.pl/pages/pytania-i-odpowiedzi"
SHOP_NAME = "Mebligo"
CSV_PATH = "faq_dataset.csv"   # wspólna baza dla wielu sklepów/URL-i


def get_soup_with_selenium(url: str) -> BeautifulSoup:
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, timeout=15)

    try:
        driver.get(url)
        time.sleep(2)

        try:
            accordion_panels = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[id^='ac-']")
                )
            )
            print(f"Znaleziono {len(accordion_panels)} paneli akordeonu (Selenium)")
        except TimeoutException:
            print("Akordeon nie został znaleziony, kontynuuję bez niego")

        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def extract_question(block) -> str:
    trigger = block.find("button", id=re.compile(r"^ac-trigger-\d+$"))
    if trigger:
        return clean_text(trigger.get_text(" ", strip=True)).rstrip(" +").strip()
    heading = block.find("h2") or block.find("h3")
    if heading:
        return clean_text(heading.get_text(" ", strip=True))
    return ""


def extract_answer(block) -> str:
    """
    Logika:
    - div#ac-panel-N -> tekst z <p> (deduplikacja), fallback: cały panel
    - jeśli brak panelu -> cały block bez <button>
    """
    panel = block.find("div", id=re.compile(r"^ac-panel-\d+$"))
    if panel:
        parts: List[str] = []
        for p in panel.find_all("p"):
            t = clean_text(p.get_text(" ", strip=True))
            if t:
                parts.append(t)
        if parts:
            seen = set()
            uniq = []
            for t in parts:
                if t not in seen:
                    seen.add(t)
                    uniq.append(t)
            return " ".join(uniq)
        return clean_text(panel.get_text(" ", strip=True))

    block_copy = BeautifulSoup(str(block), "html.parser")
    for btn in block_copy.find_all("button"):
        btn.decompose()
    return clean_text(block_copy.get_text(" ", strip=True))


def debug_panel_full(panel, idx: int):
    """
    Pełna diagnostyka JEDNEGO panelu.
    """
    panel_id = panel.get("id", "")
    print("\n" + "=" * 80)
    print(f"[DEBUG FULL] PANEL {idx}  |  id={panel_id}")
    print("=" * 80)

    raw_html = str(panel)
    print("\n[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):")
    max_len = 2000
    print(raw_html[:max_len] + ("..." if len(raw_html) > max_len else ""))

    full_text = clean_text(panel.get_text(" ", strip=True))
    print("\n[FULL TEXT Z PANELU]:")
    print(full_text)

    div_p_texts: List[str] = []
    for node in panel.find_all(["div", "p"]):
        t = clean_text(node.get_text(" ", strip=True))
        if t:
            div_p_texts.append(t)
    print("\n[SUMA TEKSTÓW z <div>/<p>]:")
    print(" | ".join(div_p_texts) if div_p_texts else "(brak)")

    panel_copy = BeautifulSoup(str(panel), "html.parser")
    for btn in panel_copy.find_all("button"):
        btn.decompose()
    no_button_text = clean_text(panel_copy.get_text(" ", strip=True))
    print("\n[TEKST PANELU bez <button>]:")
    print(no_button_text if no_button_text else "(brak)")

    final_question = extract_question(panel)
    final_answer = extract_answer(panel)

    print("\n[AKTUALNA INTERPRETACJA]:")
    print(f"Q: {final_question}")
    print(f"A: {final_answer}")

    print("\n" + "-" * 80)
    print("[KONIEC PEŁNEGO DEBUGU PANELU]")
    print("-" * 80 + "\n")


def scrape_mebligo(url: str = URL, shop: str = SHOP_NAME,
                   debug_sample: int = 5) -> List[Dict]:
    """
    1) Diagnostyka: losowe debug_sample paneli.
    2) Zwraca listę Q&A z tej jednej strony.
    """
    soup = get_soup_with_selenium(url)

    blocks = soup.find_all("div", id=re.compile(r"^ac-\d+$"))
    total = len(blocks)
    print(f"\nZnaleziono {total} bloków akordeonu (ac-0, ac-1, ...)")

    if total == 0:
        print("Brak bloków do przetworzenia.")
        return []

    # losowe panele do debug
    sample_size = min(debug_sample, total)
    indices = sorted(random.sample(range(1, total + 1), k=sample_size))
    print(f"\nPanele wybrane do PEŁNEJ DIAGNOSTYKI: {indices}")

    for idx in indices:
        panel = blocks[idx - 1]
        debug_panel_full(panel, idx)

    # właściwy scraping
    qas: List[Dict] = []
    for idx, block in enumerate(blocks, start=1):
        block_id = block.get("id", "")

        question = extract_question(block)
        answer = extract_answer(block)

        print(f"Panel {idx}: id={block_id}")
        print(f"  Q: {question[:70] + '...' if question else 'BRAK'}")
        print(f"  A_len: {len(answer)}")

        if question and answer:
            qas.append(
                {
                    "shop": shop,
                    "source_url": url,
                    "question": question,
                    "answer": answer,
                    "question_index": idx,
                    "panel_id": block_id,
                    "position": f"accordion[{idx}]",
                }
            )

    print(f"\n{shop}: {len(qas)} Q&A zebrane z {url}")
    return qas


def append_new_unique_rows(
    new_rows: List[Dict],
    csv_path: str = CSV_PATH,
    key_cols=("shop", "source_url", "question"),
) -> None:
    """
    Zasady:
    - jeśli CSV nie istnieje -> tworzy nowy plik z new_rows
    - jeśli istnieje:
        - wczytuje istniejącą bazę,
        - WYZNACZA, które new_rows mają już odpowiednik
          o tym samym (shop, source_url, question) -> te pomija,
        - dopisuje TYLKO nowe, unikalne rekordy.
    """
    if not new_rows:
        print("Brak nowych danych do zapisania.")
        return

    new_df = pd.DataFrame(new_rows)

    if not os.path.exists(csv_path):
        new_df.to_csv(csv_path, index=False, encoding="utf-8")
        print(f"Stworzono nowy plik CSV: {csv_path}")
        print(f"Dopisano {len(new_df)} nowych rekordów.")
        return

    existing_df = pd.read_csv(csv_path)

    # klucz logiczny w istniejącej bazie
    existing_keys = set(
        tuple(row[col] for col in key_cols)
        for _, row in existing_df.iterrows()
    )

    # filtruj tylko rekordy, których klucza jeszcze nie ma
    mask_new = []
    for _, row in new_df.iterrows():
        key = tuple(row[col] for col in key_cols)
        mask_new.append(key not in existing_keys)

    new_unique_df = new_df[mask_new]

    if new_unique_df.empty:
        print("Brak naprawdę nowych rekordów – CSV pozostaje bez zmian.")
        return

    updated = pd.concat([existing_df, new_unique_df], ignore_index=True)
    updated.to_csv(csv_path, index=False, encoding="utf-8")

    print(f"Zaktualizowano plik CSV: {csv_path}")
    print(f"Dopisano {len(new_unique_df)} nowych rekordów.")
    print(f"Łącznie rekordów w CSV: {len(updated)}")


if __name__ == "__main__":
    # 1) scrap jednej strony (Mebligo)
    qas = scrape_mebligo(URL, SHOP_NAME, debug_sample=5)

    # 2) dopisz tylko nowe, unikalne rekordy do wspólnej bazy CSV
    append_new_unique_rows(qas, CSV_PATH)

    print(f"\nŁącznie Q&A w tym runie: {len(qas)}")

# Nie działa prawidłowo

In [20]:
import re
import time
import random
from typing import List, Dict, Callable

import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd
import os


CSV_PATH = "faq_dataset.csv"   # wspólna baza dla wielu sklepów/URL-i


# ==========================
#  WARSTWA OGÓLNA (SELENIUM)
# ==========================

def get_soup_with_selenium(url: str) -> BeautifulSoup:
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, timeout=15)

    try:
        driver.get(url)
        time.sleep(2)

        try:
            # to jest tylko ogólny „ping” – nie zakładamy struktury
            wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "body")
                )
            )
            print("Strona załadowana (Selenium).")
        except TimeoutException:
            print("Timeout przy ładowaniu strony, kontynuuję z aktualnym HTML.")

        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


# =====================================
#  STRATEGIA PARSOWANIA DLA MEBLIGO
# =====================================

def mebligo_find_blocks(soup: BeautifulSoup):
    """Zwraca listę bloków FAQ dla Mebligo (akordeony ac-*)."""
    return soup.find_all("div", id=re.compile(r"^ac-\d+$"))


def mebligo_extract_question(block) -> str:
    trigger = block.find("button", id=re.compile(r"^ac-trigger-\d+$"))
    if trigger:
        return clean_text(trigger.get_text(" ", strip=True)).rstrip(" +").strip()
    heading = block.find("h2") or block.find("h3")
    if heading:
        return clean_text(heading.get_text(" ", strip=True))
    return ""


def mebligo_extract_answer(block) -> str:
    panel = block.find("div", id=re.compile(r"^ac-panel-\d+$"))
    if panel:
        parts: List[str] = []
        for p in panel.find_all("p"):
            t = clean_text(p.get_text(" ", strip=True))
            if t:
                parts.append(t)
        if parts:
            seen = set()
            uniq = []
            for t in parts:
                if t not in seen:
                    seen.add(t)
                    uniq.append(t)
            return " ".join(uniq)
        return clean_text(panel.get_text(" ", strip=True))

    # fallback: cały block bez buttonów
    block_copy = BeautifulSoup(str(block), "html.parser")
    for btn in block_copy.find_all("button"):
        btn.decompose()
    return clean_text(block_copy.get_text(" ", strip=True))


# ====================================
#  STRATEGIA PARSOWANIA DLA Selsey.pl
#  (na razie placeholder pod Twoją analizę)
# ====================================

def selsey_find_blocks(soup: BeautifulSoup):
    """
    TODO: dopasuj pod realną strukturę Selsey.
    Tu tylko przykład: pytania w .accordion__item
    """
    return soup.find_all("div", class_=re.compile(r"accordion", re.I))


def selsey_extract_question(block) -> str:
    """
    TODO: dopasuj selektory dla Selsey.
    Przykład: nagłówek pytania w button / h3.
    """
    btn = block.find(["button", "h3", "h2"])
    if btn:
        return clean_text(btn.get_text(" ", strip=True))
    return ""


def selsey_extract_answer(block) -> str:
    """
    TODO: dopasuj selektory dla Selsey.
    Przykład: odpowiedź w div z klasą 'accordion__content'.
    """
    content = block.find("div", class_=re.compile(r"content|answer|body", re.I))
    if content:
        return clean_text(content.get_text(" ", strip=True))
    return ""


# ===========================
#  DEBUG (wspólny mechanizm)
# ===========================

def debug_panel_full(block, idx: int,
                     extract_question: Callable,
                     extract_answer: Callable):
    panel_id = block.get("id", "") or "(brak id)"
    print("\n" + "=" * 80)
    print(f"[DEBUG FULL] BLOK {idx}  |  id={panel_id}")
    print("=" * 80)

    raw_html = str(block)
    print("\n[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):")
    max_len = 2000
    print(raw_html[:max_len] + ("..." if len(raw_html) > max_len else ""))

    full_text = clean_text(block.get_text(" ", strip=True))
    print("\n[FULL TEXT Z BLOKU]:")
    print(full_text)

    div_p_texts: List[str] = []
    for node in block.find_all(["div", "p"]):
        t = clean_text(node.get_text(" ", strip=True))
        if t:
            div_p_texts.append(t)
    print("\n[SUMA TEKSTÓW z <div>/<p>]:")
    print(" | ".join(div_p_texts) if div_p_texts else "(brak)")

    block_copy = BeautifulSoup(str(block), "html.parser")
    for btn in block_copy.find_all("button"):
        btn.decompose()
    no_button_text = clean_text(block_copy.get_text(" ", strip=True))
    print("\n[TEKST BLOKU bez <button>]:")
    print(no_button_text if no_button_text else "(brak)")

    final_question = extract_question(block)
    final_answer = extract_answer(block)

    print("\n[AKTUALNA INTERPRETACJA]:")
    print(f"Q: {final_question}")
    print(f"A: {final_answer}")

    print("\n" + "-" * 80)
    print("[KONIEC PEŁNEGO DEBUGU BLOKU]")
    print("-" * 80 + "\n")


# ===========================
#  OGÓLNY SCRAPER DLA SKLEPU
# ===========================

def scrape_shop(
    url: str,
    shop_name: str,
    find_blocks: Callable[[BeautifulSoup], List] ,
    extract_question: Callable,
    extract_answer: Callable,
    debug_sample: int = 5,
) -> List[Dict]:
    """
    Ogólna funkcja scrapująca:
    - używa podanych strategii find_blocks / extract_*
    - robi debug_sample losowych bloków
    - zwraca listę Q&A do wrzucenia do CSV
    """
    soup = get_soup_with_selenium(url)

    blocks = find_blocks(soup)
    total = len(blocks)
    print(f"\n[{shop_name}] Znaleziono {total} bloków FAQ na {url}")

    if total == 0:
        print("Brak bloków do przetworzenia.")
        return []

    sample_size = min(debug_sample, total)
    indices = sorted(random.sample(range(1, total + 1), k=sample_size))
    print(f"\nBloki wybrane do PEŁNEJ DIAGNOSTYKI: {indices}")

    for idx in indices:
        block = blocks[idx - 1]
        debug_panel_full(block, idx, extract_question, extract_answer)

    qas: List[Dict] = []
    for idx, block in enumerate(blocks, start=1):
        block_id = block.get("id", "") or ""

        question = extract_question(block)
        answer = extract_answer(block)

        print(f"Blok {idx}: id={block_id}")
        print(f"  Q: {question[:70] + '...' if question else 'BRAK'}")
        print(f"  A_len: {len(answer)}")

        if question and answer:
            qas.append(
                {
                    "shop": shop_name,
                    "source_url": url,
                    "question": question,
                    "answer": answer,
                    "question_index": idx,
                    "block_id": block_id,
                    "position": f"block[{idx}]",
                }
            )

    print(f"\n{shop_name}: {len(qas)} Q&A zebrane z {url}")
    return qas


# ===========================
#  ZAPIS DO CSV (append-only)
# ===========================

def append_new_unique_rows(
    new_rows: List[Dict],
    csv_path: str = CSV_PATH,
    key_cols=("shop", "source_url", "question"),
) -> None:
    if not new_rows:
        print("Brak nowych danych do zapisania.")
        return

    new_df = pd.DataFrame(new_rows)

    if not os.path.exists(csv_path):
        new_df.to_csv(csv_path, index=False, encoding="utf-8")
        print(f"Stworzono nowy plik CSV: {csv_path}")
        print(f"Dopisano {len(new_df)} nowych rekordów.")
        return

    existing_df = pd.read_csv(csv_path)

    existing_keys = set(
        tuple(row[col] for col in key_cols)
        for _, row in existing_df.iterrows()
    )

    mask_new = []
    for _, row in new_df.iterrows():
        key = tuple(row[col] for col in key_cols)
        mask_new.append(key not in existing_keys)

    new_unique_df = new_df[mask_new]

    if new_unique_df.empty:
        print("Brak naprawdę nowych rekordów – CSV pozostaje bez zmian.")
        return

    updated = pd.concat([existing_df, new_unique_df], ignore_index=True)
    updated.to_csv(csv_path, index=False, encoding="utf-8")

    print(f"Zaktualizowano plik CSV: {csv_path}")
    print(f"Dopisano {len(new_unique_df)} nowych rekordów.")
    print(f"Łącznie rekordów w CSV: {len(updated)}")


# ===========================
#  GŁÓWNE WEJŚCIE
# ===========================

if __name__ == "__main__":
    # PRZYKŁAD 1: Mebligo
    # url = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"
    # shop = "Mebligo"
    # qas = scrape_shop(
    #     url=url,
    #     shop_name=shop,
    #     find_blocks=mebligo_find_blocks,
    #     extract_question=mebligo_extract_question,
    #     extract_answer=mebligo_extract_answer,
    #     debug_sample=5,
    # )

    # # PRZYKŁAD 2: Selsey (DO DOPASOWANIA SELEKTORÓW)
    # url = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"
    # shop = "Selsey"

    URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"
    SHOP_NAME = "Mebligo"
    CSV_PATH = "faq_dataset.csv"   # wspólna baza dla wielu sklepów/URL-i

    qas = scrape_shop(
        url=url,
        shop_name=shop,
        find_blocks=selsey_find_blocks,
        extract_question=selsey_extract_question,
        extract_answer=selsey_extract_answer,
        debug_sample=5,
    )

    append_new_unique_rows(qas, CSV_PATH)
    print(f"\nŁącznie Q&A w tym runie: {len(qas)}")

Strona załadowana (Selenium).

[Selsey] Znaleziono 6 bloków FAQ na https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq

Bloki wybrane do PEŁNEJ DIAGNOSTYKI: [1, 3, 4, 5, 6]

[DEBUG FULL] BLOK 1  |  id=(brak id)

[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):
<div class="accordion-container-product">
<div class="ac js-enabled" id="ac-0">
<h2 class="ac-header"><button aria-controls="ac-panel-0" aria-disabled="false" aria-expanded="false" class="ac-trigger" id="ac-trigger-0" role="button" type="button">Czy można gdzieś zobaczyć meble przed dostawą/usiąść na nich?</button></h2>
<div aria-labelledby="ac-trigger-0" class="ac-panel" id="ac-panel-0" role="region" style="transition-duration: 600ms; height: 0px;">
<p>Niestety, nie posiadamy stacjonarnych salonów sprzedaży.</p>
</div>
</div>
<div class="ac js-enabled" id="ac-1">
<h2 class="ac-header"><button aria-controls="ac-panel-1" aria-disabled="false" aria-expanded="false" class="ac-trigger" id="ac-trigger-1" role="button" type

# Nie działa prawidłowo

In [24]:
import re
import time
import random
from typing import List, Dict
from urllib.parse import urlparse

import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd
import os


CSV_PATH = "faq_dataset.csv"  # wspólna baza dla wielu sklepów/URL-i


def get_shop_name_from_url(url: str) -> str:
    """
    Proste wyciąganie nazwy sklepu z domeny:
    https://mebligo.pl/...   -> 'mebligo'
    https://www.selsey.pl... -> 'selsey'
    """
    parsed = urlparse(url)
    host = parsed.netloc  # np. 'www.selsey.pl'
    if host.startswith("www."):
        host = host[4:]
    # bierzemy pierwszy fragment przed kropką
    shop = host.split(".")[0]
    return shop.capitalize()  # 'mebligo' -> 'Mebligo', 'selsey' -> 'Selsey'


def get_soup_with_selenium(url: str) -> BeautifulSoup:
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, timeout=15)

    try:
        driver.get(url)
        time.sleep(2)

        try:
            accordion_panels = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[id^='ac-']")
                )
            )
            print(f"Znaleziono {len(accordion_panels)} paneli akordeonu (Selenium)")
        except TimeoutException:
            print("Akordeon nie został znaleziony, kontynuuję bez niego")

        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def extract_question(block) -> str:
    trigger = block.find("button", id=re.compile(r"^ac-trigger-\d+$"))
    if trigger:
        return clean_text(trigger.get_text(" ", strip=True)).rstrip(" +").strip()
    heading = block.find("h2") or block.find("h3")
    if heading:
        return clean_text(heading.get_text(" ", strip=True))
    return ""


def extract_answer(block) -> str:
    """
    Logika:
    - div#ac-panel-N -> tekst z <p> (deduplikacja), fallback: cały panel
    - jeśli brak panelu -> cały block bez <button>
    """
    panel = block.find("div", id=re.compile(r"^ac-panel-\d+$"))
    if panel:
        parts: List[str] = []
        for p in panel.find_all("p"):
            t = clean_text(p.get_text(" ", strip=True))
            if t:
                parts.append(t)
        if parts:
            seen = set()
            uniq = []
            for t in parts:
                if t not in seen:
                    seen.add(t)
                    uniq.append(t)
            return " ".join(uniq)
        return clean_text(panel.get_text(" ", strip=True))

    block_copy = BeautifulSoup(str(block), "html.parser")
    for btn in block_copy.find_all("button"):
        btn.decompose()
    return clean_text(block_copy.get_text(" ", strip=True))


def debug_panel_full(panel, idx: int):
    panel_id = panel.get("id", "")
    print("\n" + "=" * 80)
    print(f"[DEBUG FULL] PANEL {idx}  |  id={panel_id}")
    print("=" * 80)

    raw_html = str(panel)
    print("\n[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):")
    max_len = 2000
    print(raw_html[:max_len] + ("..." if len(raw_html) > max_len else ""))

    full_text = clean_text(panel.get_text(" ", strip=True))
    print("\n[FULL TEXT Z PANELU]:")
    print(full_text)

    div_p_texts: List[str] = []
    for node in panel.find_all(["div", "p"]):
        t = clean_text(node.get_text(" ", strip=True))
        if t:
            div_p_texts.append(t)
    print("\n[SUMA TEKSTÓW z <div>/<p>]:")
    print(" | ".join(div_p_texts) if div_p_texts else "(brak)")

    panel_copy = BeautifulSoup(str(panel), "html.parser")
    for btn in panel_copy.find_all("button"):
        btn.decompose()
    no_button_text = clean_text(panel_copy.get_text(" ", strip=True))
    print("\n[TEKST PANELU bez <button>]:")
    print(no_button_text if no_button_text else "(brak)")

    final_question = extract_question(panel)
    final_answer = extract_answer(panel)

    print("\n[AKTUALNA INTERPRETACJA]:")
    print(f"Q: {final_question}")
    print(f"A: {final_answer}")

    print("\n" + "-" * 80)
    print("[KONIEC PEŁNEGO DEBUGU PANELU]")
    print("-" * 80 + "\n")


def scrape_page(url: str, shop: str, debug_sample: int = 5) -> List[Dict]:
    """
    1) Diagnostyka: losowe debug_sample paneli.
    2) Zwraca listę Q&A z tej jednej strony.
    """
    soup = get_soup_with_selenium(url)

    blocks = soup.find_all("div", id=re.compile(r"^ac-\d+$"))
    total = len(blocks)
    print(f"\n[{shop}] Znaleziono {total} bloków akordeonu (ac-0, ac-1, ...) na {url}")

    if total == 0:
        print("Brak bloków do przetworzenia.")
        return []

    sample_size = min(debug_sample, total)
    indices = sorted(random.sample(range(1, total + 1), k=sample_size))
    print(f"\nPanele wybrane do PEŁNEJ DIAGNOSTYKI: {indices}")

    for idx in indices:
        panel = blocks[idx - 1]
        debug_panel_full(panel, idx)

    qas: List[Dict] = []
    for idx, block in enumerate(blocks, start=1):
        block_id = block.get("id", "")

        question = extract_question(block)
        answer = extract_answer(block)

        print(f"Panel {idx}: id={block_id}")
        print(f"  Q: {question[:70] + '...' if question else 'BRAK'}")
        print(f"  A_len: {len(answer)}")

        if question and answer:
            qas.append(
                {
                    "shop": shop,
                    "source_url": url,
                    "question": question,
                    "answer": answer,
                    "question_index": idx,
                    "panel_id": block_id,
                    "position": f"accordion[{idx}]",
                }
            )

    print(f"\n{shop}: {len(qas)} Q&A zebrane z {url}")
    return qas


def append_new_unique_rows(
    new_rows: List[Dict],
    csv_path: str = CSV_PATH,
    key_cols=("shop", "source_url", "question"),
) -> None:
    if not new_rows:
        print("Brak nowych danych do zapisania.")
        return

    new_df = pd.DataFrame(new_rows)

    if not os.path.exists(csv_path):
        new_df.to_csv(csv_path, index=False, encoding="utf-8")
        print(f"Stworzono nowy plik CSV: {csv_path}")
        print(f"Dopisano {len(new_df)} nowych rekordów.")
        return

    existing_df = pd.read_csv(csv_path)

    existing_keys = set(
        tuple(row[col] for col in key_cols)
        for _, row in existing_df.iterrows()
    )

    mask_new = []
    for _, row in new_df.iterrows():
        key = tuple(row[col] for col in key_cols)
        mask_new.append(key not in existing_keys)

    new_unique_df = new_df[mask_new]

    if new_unique_df.empty:
        print("Brak naprawdę nowych rekordów – CSV pozostaje bez zmian.")
        return

    updated = pd.concat([existing_df, new_unique_df], ignore_index=True)
    updated.to_csv(csv_path, index=False, encoding="utf-8")

    print(f"Zaktualizowano plik CSV: {csv_path}")
    print(f"Dopisano {len(new_unique_df)} nowych rekordów.")
    print(f"Łącznie rekordów w CSV: {len(updated)}")


if __name__ == "__main__":
    # 1) Pytamy o URL w konsoli
    url = input("Podaj URL strony FAQ do zeskrapowania: ").strip()
    if not url:
        raise SystemExit("Nie podano URL – kończę.")

    # 2) Automatycznie wyznaczamy SHOP_NAME z adresu
    shop_name = get_shop_name_from_url(url)
    print(f"Rozpoznany shop_name: {shop_name}")

    # 3) Scraping i zapis
    qas = scrape_page(url, shop_name, debug_sample=5)
    append_new_unique_rows(qas, CSV_PATH)

    print(f"\nŁącznie Q&A w tym runie: {len(qas)}")

Rozpoznany shop_name: Maxfliz
Akordeon nie został znaleziony, kontynuuję bez niego

[Maxfliz] Znaleziono 0 bloków akordeonu (ac-0, ac-1, ...) na https://www.maxfliz.pl/najczesciej-zadawane-pytania
Brak bloków do przetworzenia.
Brak nowych danych do zapisania.

Łącznie Q&A w tym runie: 0


# Nie działa prawidłowo

In [26]:
import re
import time
import random
from typing import List, Dict, Callable
from urllib.parse import urlparse

import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd
import os


CSV_PATH = "faq_dataset.csv"


def get_shop_name_from_url(url: str) -> str:
    parsed = urlparse(url)
    host = parsed.netloc
    if host.startswith("www."):
        host = host[4:]
    shop = host.split(".")[0]
    return shop.capitalize()


def get_domain(url: str) -> str:
    parsed = urlparse(url)
    host = parsed.netloc
    if host.startswith("www."):
        host = host[4:]
    return host  # np. 'mebligo.pl', 'selsey.pl', 'maxfliz.pl'


def get_soup_with_selenium(url: str) -> BeautifulSoup:
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, timeout=15)

    try:
        driver.get(url)
        time.sleep(2)

        try:
            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "body")))
            print("Strona załadowana (Selenium).")
        except TimeoutException:
            print("Timeout przy ładowaniu strony, kontynuuję z aktualnym HTML.")

        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


# ---------- STRATEGIA 1: akordeony ac-* (Mebligo, Selsey) ----------

def find_ac_blocks(soup: BeautifulSoup):
    return soup.find_all("div", id=re.compile(r"^ac-\d+$"))


def extract_question_ac(block) -> str:
    trigger = block.find("button", id=re.compile(r"^ac-trigger-\d+$"))
    if trigger:
        return clean_text(trigger.get_text(" ", strip=True)).rstrip(" +").strip()
    heading = block.find("h2") or block.find("h3")
    if heading:
        return clean_text(heading.get_text(" ", strip=True))
    return ""


def extract_answer_ac(block) -> str:
    panel = block.find("div", id=re.compile(r"^ac-panel-\d+$"))
    if panel:
        parts: List[str] = []
        for p in panel.find_all("p"):
            t = clean_text(p.get_text(" ", strip=True))
            if t:
                parts.append(t)
        if parts:
            seen = set()
            uniq = []
            for t in parts:
                if t not in seen:
                    seen.add(t)
                    uniq.append(t)
            return " ".join(uniq)
        return clean_text(panel.get_text(" ", strip=True))

    block_copy = BeautifulSoup(str(block), "html.parser")
    for btn in block_copy.find_all("button"):
        btn.decompose()
    return clean_text(block_copy.get_text(" ", strip=True))


# ---------- STRATEGIA 2: ogólna diagnostyczna (Maxfliz i inne) ----------

def find_generic_blocks(soup: BeautifulSoup):
    """
    Tymczasowo: bierzemy wszystkie sekcje/nagłówki, żebyś mógł obejrzeć strukturę.
    Potem na podstawie debug outputu dopiszesz dedykowaną strategię.
    """
    # przykład: sekcje FAQ mogą być w article, section, div[class~=faq]
    blocks = soup.find_all(["section", "article", "div"], class_=re.compile(r"faq", re.I))
    if blocks:
        return blocks
    # fallback: duże sekcje bez klasy
    return soup.find_all("section")


def extract_question_generic(block) -> str:
    """
    Bardzo ogólna heurystyka:
    - szukamy h2/h3, ewentualnie silnie wyróżnione teksty z '?'
    """
    h = block.find(["h2", "h3"])
    if h:
        txt = clean_text(h.get_text(" ", strip=True))
        if "?" in txt:
            return txt
    # fallback: pierwszy akapit z '?'
    for p in block.find_all("p"):
        txt = clean_text(p.get_text(" ", strip=True))
        if "?" in txt and len(txt) < 200:
            return txt
    return ""


def extract_answer_generic(block) -> str:
    """
    Ogólna heurystyka odpowiedzi: wszystko poza pytaniem, w p/div.
    To jest tylko narzędzie diagnostyczne – docelowo zrobimy dedykowany parser.
    """
    # skopiuj blok i usuń nagłówki z pytaniami
    copy = BeautifulSoup(str(block), "html.parser")
    for h in copy.find_all(["h2", "h3"]):
        h.decompose()
    txts = []
    for p in copy.find_all("p"):
        t = clean_text(p.get_text(" ", strip=True))
        if t:
            txts.append(t)
    return " ".join(txts)


# ---------- DEBUG wspólny ----------

def debug_panel_full(block, idx: int,
                     extract_question: Callable,
                     extract_answer: Callable,
                     shop: str):
    block_id = block.get("id", "") or "(brak id)"
    print("\n" + "=" * 80)
    print(f"[DEBUG FULL] BLOK {idx}  |  id={block_id} | shop={shop}")
    print("=" * 80)

    raw_html = str(block)
    print("\n[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):")
    max_len = 2000
    print(raw_html[:max_len] + ("..." if len(raw_html) > max_len else ""))

    full_text = clean_text(block.get_text(" ", strip=True))
    print("\n[FULL TEXT Z BLOKU]:")
    print(full_text)

    div_p_texts: List[str] = []
    for node in block.find_all(["div", "p"]):
        t = clean_text(node.get_text(" ", strip=True))
        if t:
            div_p_texts.append(t)
    print("\n[SUMA TEKSTÓW z <div>/<p>]:")
    print(" | ".join(div_p_texts) if div_p_texts else "(brak)")

    copy = BeautifulSoup(str(block), "html.parser")
    for btn in copy.find_all("button"):
        btn.decompose()
    no_button_text = clean_text(copy.get_text(" ", strip=True))
    print("\n[TEKST BLOKU bez <button>]:")
    print(no_button_text if no_button_text else "(brak)")

    final_q = extract_question(block)
    final_a = extract_answer(block)

    print("\n[AKTUALNA INTERPRETACJA]:")
    print(f"Q: {final_q}")
    print(f"A: {final_a}")

    print("\n" + "-" * 80)
    print("[KONIEC PEŁNEGO DEBUGU BLOKU]")
    print("-" * 80 + "\n")


# ---------- GŁÓWNY SCRAPER (używa wybranej strategii) ----------

def scrape_with_strategy(
    url: str,
    shop: str,
    find_blocks: Callable[[BeautifulSoup], List],
    extract_question: Callable,
    extract_answer: Callable,
    debug_sample: int = 5,
) -> List[Dict]:
    soup = get_soup_with_selenium(url)

    blocks = find_blocks(soup)
    total = len(blocks)
    print(f"\n[{shop}] Znaleziono {total} bloków FAQ na {url}")

    if total == 0:
        print("Brak bloków do przetworzenia.")
        return []

    sample_size = min(debug_sample, total)
    indices = sorted(random.sample(range(1, total + 1), k=sample_size))
    print(f"\nBloki wybrane do PEŁNEJ DIAGNOSTYKI: {indices}")

    for idx in indices:
        block = blocks[idx - 1]
        debug_panel_full(block, idx, extract_question, extract_answer, shop)

    qas: List[Dict] = []
    for idx, block in enumerate(blocks, start=1):
        block_id = block.get("id", "") or ""

        q = extract_question(block)
        a = extract_answer(block)

        print(f"Blok {idx}: id={block_id}")
        print(f"  Q: {q[:70] + '...' if q else 'BRAK'}")
        print(f"  A_len: {len(a)}")

        if q and a:
            qas.append(
                {
                    "shop": shop,
                    "source_url": url,
                    "question": q,
                    "answer": a,
                    "question_index": idx,
                    "block_id": block_id,
                    "position": f"block[{idx}]",
                }
            )

    print(f"\n{shop}: {len(qas)} Q&A zebrane z {url}")
    return qas


def append_new_unique_rows(
    new_rows: List[Dict],
    csv_path: str = CSV_PATH,
    key_cols=("shop", "source_url", "question"),
) -> None:
    if not new_rows:
        print("Brak nowych danych do zapisania.")
        return

    new_df = pd.DataFrame(new_rows)

    if not os.path.exists(csv_path):
        new_df.to_csv(csv_path, index=False, encoding="utf-8")
        print(f"Stworzono nowy plik CSV: {csv_path}")
        print(f"Dopisano {len(new_df)} nowych rekordów.")
        return

    existing_df = pd.read_csv(csv_path)

    existing_keys = set(
        tuple(row[col] for col in key_cols)
        for _, row in existing_df.iterrows()
    )

    mask_new = []
    for _, row in new_df.iterrows():
        key = tuple(row[col] for col in key_cols)
        mask_new.append(key not in existing_keys)

    new_unique_df = new_df[mask_new]

    if new_unique_df.empty:
        print("Brak naprawdę nowych rekordów – CSV pozostaje bez zmian.")
        return

    updated = pd.concat([existing_df, new_unique_df], ignore_index=True)
    updated.to_csv(csv_path, index=False, encoding="utf-8")

    print(f"Zaktualizowano plik CSV: {csv_path}")
    print(f"Dopisano {len(new_unique_df)} nowych rekordów.")
    print(f"Łącznie rekordów w CSV: {len(updated)}")


if __name__ == "__main__":
    url = input("Podaj URL strony FAQ do zeskrapowania: ").strip()
    if not url:
        raise SystemExit("Nie podano URL – kończę.")

    shop = get_shop_name_from_url(url)
    domain = get_domain(url)
    print(f"Rozpoznany shop_name: {shop}, domain: {domain}")

    # wybór strategii
    if domain in {"mebligo.pl", "selsey.pl"}:
        print("Używam strategii akordeonu ac-* (Mebligo/Selsey).")
        qas = scrape_with_strategy(
            url=url,
            shop=shop,
            find_blocks=find_ac_blocks,
            extract_question=extract_question_ac,
            extract_answer=extract_answer_ac,
            debug_sample=5,
        )
    else:
        print("Używam ogólnej strategii diagnostycznej (brak ac-*).")
        qas = scrape_with_strategy(
            url=url,
            shop=shop,
            find_blocks=find_generic_blocks,
            extract_question=extract_question_generic,
            extract_answer=extract_answer_generic,
            debug_sample=5,
        )

    append_new_unique_rows(qas, CSV_PATH)
    print(f"\nŁącznie Q&A w tym runie: {len(qas)}")

Rozpoznany shop_name: Maxfliz, domain: maxfliz.pl
Używam ogólnej strategii diagnostycznej (brak ac-*).
Strona załadowana (Selenium).

[Maxfliz] Znaleziono 1 bloków FAQ na https://www.maxfliz.pl/najczesciej-zadawane-pytania

Bloki wybrane do PEŁNEJ DIAGNOSTYKI: [1]

[DEBUG FULL] BLOK 1  |  id=(brak id) | shop=Maxfliz

[RAW HTML] (PEŁNY lub ucięty, jeśli bardzo długi):
<div class="faq-module">
<h2>Wizualizacje Wnętrz Maxfliz - Pytania i odpowiedzi</h2>
<ul class="faq-list">
<li>
<a class="faq-link accordion-title">Czy Maxfliz posiada usługę wizualizacji wnętrz?
                        <i class="icon icon-arrow-drop-down"></i>
</a>
<div class="accordion-content">
<p>Tak, Maxfliz oferuje profesjonalne usługi wizualizacji wnętrz, dzięki którym możesz zobaczyć swoją przyszłą aranżację jeszcze przed rozpoczęciem prac. W naszej ofercie znajdziesz:</p>
<ul style="list-style-type: circle;">
<li><strong>Wizualizację łazienki</strong> – realistyczne ujęcia przygotowane na podstawie wybranych płyte

In [7]:
import re
import time
from typing import List, Dict

import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup_with_selenium(url: str) -> BeautifulSoup:
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, timeout=15)

    try:
        driver.get(url)
        time.sleep(2)

        try:
            accordion_panels = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[id^='ac-']")
                )
            )
            print(f"Znaleziono {len(accordion_panels)} paneli akordeonu (Selenium)")
        except TimeoutException:
            print("Akordeon nie został znaleziony, kontynuuję bez niego")

        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def parse_mebligo_final(url: str = URL) -> List[Dict]:
    """
    Finalna wersja pod aktualną strukturę Mebligo:
    <div id="ac-N" class="ac">
        <h2 class="ac-header">
            <button id="ac-trigger-N">PYTANIE</button>
        </h2>
        <div id="ac-panel-N" class="ac-panel">
            <p>ODPOWIEDŹ</p>
        </div>
    </div>
    """
    soup = get_soup_with_selenium(url)
    qas: List[Dict] = []

    # nadrzędne bloki akordeonu
    blocks = soup.find_all("div", id=re.compile(r"^ac-\d+$"))
    print(f"Znaleziono {len(blocks)} bloków (ac-0, ac-1, ...)")

    for idx, block in enumerate(blocks, start=1):
        block_id = block.get("id", "")
        print(f"\nBlock {idx}: {block_id}")

        # 1) pytanie: button w h2.ac-header
        trigger = block.find("button", id=re.compile(r"^ac-trigger-\d+$"))
        question_text = None
        if trigger:
            question_text = clean_text(trigger.get_text(" ", strip=True))
            question_text = question_text.rstrip(" +").strip()
        else:
            # fallback – mało prawdopodobny, ale na wszelki wypadek
            header = block.find("h2", class_="ac-header") or block.find("h3")
            if header:
                question_text = clean_text(header.get_text(" ", strip=True))

        # 2) odpowiedź: div.ac-panel pod tym samym blockiem
        panel = block.find("div", id=re.compile(r"^ac-panel-\d+$"))
        answer_text = ""
        if panel:
            # weź tekst tylko z p/div wewnątrz panelu
            answer_parts = []
            for node in panel.find_all(["p", "div"]):
                t = clean_text(node.get_text(" ", strip=True))
                if t:
                    answer_parts.append(t)
            # jeżeli nic nie znaleziono, fallback: cały panel
            if not answer_parts:
                answer_text = clean_text(panel.get_text(" ", strip=True))
            else:
                # usuń duplikaty typu: panel + p (to samo)
                # najprościej: lista unikalna z zachowaniem kolejności
                seen = set()
                unique_parts = []
                for part in answer_parts:
                    if part not in seen:
                        seen.add(part)
                        unique_parts.append(part)
                answer_text = " ".join(unique_parts)
        else:
            # fallback: stara logika – panel = block bez buttona
            panel_copy = BeautifulSoup(str(block), "html.parser")
            for btn in panel_copy.find_all("button"):
                btn.decompose()
            answer_text = clean_text(panel_copy.get_text(" ", strip=True))

        print(f"  Q: {question_text[:80] + '...' if question_text else 'BRAK'}")
        print(f"  A_len: {len(answer_text)}")

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": url,
                    "question": question_text,
                    "answer": answer_text,
                    "question_index": idx,
                    "panel_id": block_id,
                    "position": f"accordion[{idx}]",
                }
            )

    print(f"\nMebligo: {len(qas)} Q&A pobrane (final)")
    return qas


if __name__ == "__main__":
    r1 = parse_mebligo_final()
    print(f"\nŁącznie Q&A: {len(r1)}")
    if r1:
        print("\nPierwsze 5 pytań:")
        for i, qa in enumerate(r1[:5], start=1):
            print(f"\n{i}. {qa['question']}")
            print(f"   Odpowiedź: {qa['answer'][:200]}...")

Znaleziono 90 paneli akordeonu (Selenium)
Znaleziono 45 bloków (ac-0, ac-1, ...)

Block 1: ac-0
  Q: Czy można gdzieś zobaczyć meble przed dostawą/usiąść na nich?...
  A_len: 56

Block 2: ac-1
  Q: Czy jest możliwość modyfikacji mebla (zmiana wymiarów, koloru)?...
  A_len: 108

Block 3: ac-2
  Q: Gdzie znajdę instrukcję montażu?...
  A_len: 184

Block 4: ac-3
  Q: Jakie są wymiary paczek?...
  A_len: 161

Block 5: ac-4
  Q: Mebel będzie dostarczany w całości czy paczkach do samodzielnego montażu?...
  A_len: 259

Block 6: ac-5
  Q: Jak mogę określić stronę mebla?...
  A_len: 196

Block 7: ac-6
  Q: Jakie wypełnienie jest bardziej komfortowe?...
  A_len: 891

Block 8: ac-7
  Q: Co to znaczy, że narożnik jest uniwersalny?...
  A_len: 253

Block 9: ac-8
  Q: Jaki jest okres gwarancji?...
  A_len: 63

Block 10: ac-9
  Q: Jak zostaną zwrócone moje środki?...
  A_len: 117

Block 11: ac-10
  Q: Jak długo trzeba czekać na zwrot środków?...
  A_len: 206

Block 12: ac-11
  Q: Czy mogę zwrócić pr

In [35]:
import csv
import os

# Definicja ścieżki
BASE_DIR = r"C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA"
INPUT_CSV = os.path.join(BASE_DIR, "faq_input.csv")
OUTPUT_CSV = os.path.join(BASE_DIR, "faq_output.csv")

def save_to_csv(qas, filename):
    fieldnames = ["shop", "source_url", "question", "answer"]
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(qas)

def load_from_csv(filename):
    fieldnames = ["shop", "source_url", "question", "answer"]
    qas = []
    with open(filename, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f, fieldnames=fieldnames)
        for row in reader:
            qas.append(row)
    return qas

# Tworzenie ścieżki (jeśli nie istnieje)
os.makedirs(BASE_DIR, exist_ok=True)

# Przykładowe dane (zastąp przez parse_mebligo() jeśli potrzebujesz)
r1 = [
    {"shop": "Mebligo", "source_url": "https://mebligo.pl/faq1", "question": "Jakie są czasy dostawy?", "answer": "Dostawa w 2-3 dni robocze."},
    {"shop": "Mebligo", "source_url": "https://mebligo.pl/faq2", "question": "Czy mogę zwrócić produkt?", "answer": "Tak, w ciągu 14 dni bez wyjaśnienia przyczyny."}
]

# Zapis do faq_input.csv
save_to_csv(r1, INPUT_CSV)
print(f"✓ {INPUT_CSV} utworzony: {len(r1)} Q&A")

# Odczyt z faq_input.csv i zapis do faq_output.csv
data = load_from_csv(INPUT_CSV)
save_to_csv(data, OUTPUT_CSV)
print(f"✓ {OUTPUT_CSV} utworzony: {len(data)} Q&A")

✓ C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_input.csv utworzony: 2 Q&A
✓ C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_output.csv utworzony: 3 Q&A


In [37]:
import sys
from pathlib import Path

# Ścieżka do katalogu, gdzie leży skraper_faq.py
project_dir = Path(r"C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED")
sys.path.append(str(project_dir))

print(project_dir in map(Path, map(str, sys.path)))  # dla pewności

True


In [41]:
from skraper_faq import process_urls

process_urls("faq_input.csv", output_csv="faq_output.csv")

[INFO] INPUT:  C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_input.csv
[INFO] OUTPUT: C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_output.csv

[INFO] Przetwarzam: unknown -> https://forte.com.pl
[INFO] unknown: znaleziono 0 paneli ac-panel-*
[INFO] unknown: znaleziono 0 Q&A
[WARN] unknown: brak Q&A, nic nie zapisano

[INFO] Przetwarzam: unknown -> https://szynaka.pl/pytania-i-odpowiedzi/
[INFO] unknown: znaleziono 0 paneli ac-panel-*
[INFO] unknown: znaleziono 0 Q&A
[WARN] unknown: brak Q&A, nic nie zapisano

[INFO] Przetwarzam: unknown -> https://wersal.pl
[INFO] unknown: znaleziono 0 paneli ac-panel-*
[INFO] unknown: znaleziono 0 Q&A
[WARN] unknown: brak Q&A, nic nie zapisano

[INFO] Przetwarzam: unknown -> https://adriana.com.pl
[INFO] unknown: znaleziono 0 paneli ac-panel-*
[INFO] unknown: znaleziono 0 Q&A
[WARN] unknown: brak Q&A, nic nie zapisano

[INFO] Przetwarzam: unknown -> https://meblewojcik.pl/faq-czeste-pytania/
[INFO] unknown: znaleziono 0 paneli ac-pa

KeyboardInterrupt: 

## 2. StolarMeble.pl

**Struktura:** `div#content` → `button` (pytanie) + następny element rodzeństwo (odpowiedź). Wszystkie elementy widoczne bezpośrednio w DOM.

In [ ]:
def parse_stolar() -> list:
    url = "https://stolarmeble.pl/faq/"
    soup = get_soup(url)
    qas = []

    content = soup.find("div", id="content")
    if not content:
        return qas

    for btn in content.find_all("button"):
        question = btn.get_text(" ", strip=True)
        if not question:
            continue
        answer_node = btn.find_next_sibling()
        if not answer_node:
            answer_node = btn.parent.find_next_sibling()
        answer = answer_node.get_text(" ", strip=True) if answer_node else ""

        if question and answer:
            qas.append({"shop": "Stolar Meble", "source_url": url,
                        "question": question, "answer": answer})
    return qas

r2 = parse_stolar()
print(f"Stolar: {len(r2)} Q&A")
r2[:2]

## 3. MeblujemyDOM.pl

**Struktura:** `div#content` → naprzemienne bloki tekstowe: element z `?` = pytanie, kolejne bloki do następnego `?` = odpowiedź.

In [ ]:
def parse_meblujemydom() -> list:
    url = "https://meblujemydom.pl/pl/help/faq-najczesciej-zadawane-pytania-2"
    soup = get_soup(url)
    qas = []

    content = soup.find("div", id="content")
    if not content:
        return qas

    children = [c for c in content.children
                if hasattr(c, "name") and c.name is not None]
    
    i = 0
    while i < len(children):
        node = children[i]
        text = node.get_text(" ", strip=True)
        is_toc_link = bool(node.find("a")) and re.match(r"^\d+\.", text)
        is_question = (
            "?" in text
            and not is_toc_link
            and 10 < len(text) < 300
            and not node.find("a", href=True)
        )
        if is_question:
            answer_parts = []
            j = i + 1
            while j < len(children):
                next_text = children[j].get_text(" ", strip=True)
                if "?" in next_text and len(next_text) < 300:
                    break
                if next_text:
                    answer_parts.append(next_text)
                j += 1
            answer = " ".join(answer_parts).strip()
            if text and answer:
                qas.append({"shop": "MeblujemyDOM", "source_url": url,
                            "question": text, "answer": answer})
            i = j
        else:
            i += 1
    return qas

r3 = parse_meblujemydom()
print(f"MeblujemyDOM: {len(r3)} Q&A")
r3[:2]

## 4. SalonMeblowy.net.pl

**Struktura:** `div#afaq` → `h2`/`h3` = pytanie, kolejne `p`/`div`/`generic` do następnego nagłówka = odpowiedź.

In [ ]:
def parse_salonmeblowy() -> list:
    url = "https://www.salonmeblowy.net.pl/faq.ehtml"
    soup = get_soup(url)
    qas = []

    afaq = soup.find("div", id="afaq")
    if not afaq:
        return qas

    children = [c for c in afaq.children
                if hasattr(c, "name") and c.name is not None]

    i = 0
    while i < len(children):
        node = children[i]
        if node.name in ("h2", "h3"):
            question = node.get_text(" ", strip=True)
            answer_parts = []
            j = i + 1
            while j < len(children):
                sibling = children[j]
                if sibling.name in ("h2", "h3"):
                    break
                text = sibling.get_text(" ", strip=True)
                if text:
                    answer_parts.append(text)
                j += 1
            answer = " ".join(answer_parts).strip()
            if question and answer:
                qas.append({"shop": "SalonMeblowy.net", "source_url": url,
                            "question": question, "answer": answer})
            i = j
        else:
            i += 1
    return qas

r4 = parse_salonmeblowy()
print(f"SalonMeblowy: {len(r4)} Q&A")
r4[:2]

## 5. MebleM4.pl

**Struktura:** elementy pasujące do regex `^\d+\.\s+` = pytanie, kolejne bloki tekstowe do następnego numeru = odpowiedź.

In [ ]:
def parse_meblem4() -> list:
    url = "https://meblem4.pl/faq-najczesciej-zadawane-pytania-w-meblem4-pl,p39.html"
    try:
        soup = get_soup(url)
    except:
        return []
    qas = []

    # Najszerszy kontener, w którym może znajdować się treść artykułu
    content = soup.find("div", id="box_article")
    if not content:
        return qas

    elements = content.find_all(['p', 'div', 'span'])
    
    current_q = None
    current_a = []
    
    for el in elements:
        text = el.get_text(" ", strip=True)
        if not text:
            continue
            
        # Identyfikacja pytania po numeracji (np. '1. ', '2. ')
        if re.match(r"^\d+\.\s+", text) and len(text) < 200:
            if current_q and current_a:
                qas.append({"shop": "MebleM4", "source_url": url, 
                            "question": current_q, "answer": " ".join(current_a).strip()})
            current_q = text
            current_a = []
        elif current_q:
            # Dodawanie kolejnych bloków jako odpowiedź
            current_a.append(text)
            
    if current_q and current_a:
        qas.append({"shop": "MebleM4", "source_url": url, 
                    "question": current_q, "answer": " ".join(current_a).strip()})
        
    return qas

r5 = parse_meblem4()
print(f"MebleM4: {len(r5)} Q&A")
r5[:2]

## Podsumowanie i zebranie danych do DataFrame

In [ ]:
all_qas = r1 + r2 + r3 + r4 + r5
df = pd.DataFrame(all_qas)
print(f"Zebrano łącznie: {len(df)} par Q&A z 5 sklepów.")
df.head()